# Advanced Python

[合集·不基础的python基础 - 码农高天](https://space.bilibili.com/245645656/channel/collectiondetail?sid=346060)




## [迭代器 & for loop](https://www.bilibili.com/video/BV1ca411t7A9/)


In [1]:
l = [1,3,5]
for i in l:
    print(i)

d = {"a":1, "b":2}
for i in d:
    print(i)


1
3
5
a
b


[iterable -- 可迭代对象](https://docs.python.org/zh-cn/3/glossary.html#term-iterable)

一种能够逐个返回其成员项的对象。 可迭代对象的例子包括所有序列类型（如 list, str 和 tuple 等）以及某些非序列类型如 dict, 文件对象 以及任何自定义类的对象，只要满足
1. 定义了 \_\_iter\_\_() 方法 - 返回一个 iterator
2. 或实现了 sequence 语义的 \_\_getitem\_\_() 方法

可迭代对象可被用于 for 循环以及许多其他需要一个序列的地方

[iterator -- 迭代器](https://docs.python.org/zh-cn/3/glossary.html#term-iterator)

用来表示一连串数据流的对象。 重复调用迭代器的 \_\_next\_\_() 方法 (或将其传给内置函数 next()) 将逐个返回流中的项。 当没有数据可用时则将引发 StopIteration 异常

迭代器必须具有 \_\_iter\_\_() 方法用来返回该迭代器对象自身，**因此迭代器必定也是可迭代对象**，可被用于其他可迭代对象适用的大部分场合


两者区别
1. iterable
   1. 类似于数据的保存着
   2. 可以没有状态
   3. 可以不知道 iterator 数到哪里了
   4. 需要有能力产生 iterator
2. iterator
   1. 一定是有状态的
   2. 不需要interface 修改 iterable 里面的数据

In [24]:
# 链表实现

class NodeIter:
    def __init__(self, node) -> None:
        self.curNode = node
    def __next__(self) -> None:
        if self.curNode is None:
            raise StopIteration
        node = self.curNode
        self.curNode = self.curNode.next
        return node
    def __iter__(self):
        return self

class Node:
    def __init__(self, num) -> None:
        self.num = num
        self.next = None
        
    def __iter__(self):
        return NodeIter(self)


In [23]:
node1 = Node(1)
node2 = Node(2)
node3 = Node(3)

node1.next = node2
node2.next = node3
# node3.next = node1  # 别尝试

for node in node1:
    print(node.num)

print("--------------")

for node in iter(node1):
    print(node.num)


1
2
3
--------------


In [25]:
from collections.abc import Iterable, Iterator

testNode = Node(0)
testNodeIter = NodeIter(node)

print(isinstance(testNode,Iterable))  # True
print(isinstance(testNode,Iterator))  # False
print(isinstance(testNodeIter,Iterable))  # True
print(isinstance(testNodeIter,Iterator))  # True


True
False
True
True


## [生成器](https://www.bilibili.com/video/BV1KS4y1D7Qb/)

生成器是一种特殊的迭代器

两个概念
1. 生成器函数
2. 生成器对象

当编译时发现函数的定义里有 yield 关键词时，则不会将函数当做普通函数处理，而是当做**生成器函数**

调用**生成器函数**得到一个**生成器对象**

yield、return 都不是返回值

对生成器对象使用 next 函数时，才开始真正运行函数本体

生成器函数会先执行 yield 之前的部分，并在 yield 的地方返回 值，然后函数暂停执行

下次再对生成器对象使用next的时候，会接着 yield 运行下面的代码

在生成器函数内，return 等价于 raise StopIteration

In [30]:
def gen(num):  # 生成器函数
    while num>0:
        yield num
        num -= 1
    return

g = gen(5)  # 生成器对象
print(next(g))

print("-----")

for i in g:
    print(i)


5
-----
4
3
2
1


In [33]:
class Node:
    def __init__(self, num) -> None:
        self.num = num
        self.next = None
    
    def __iter__(self):
        node = self
        while node is not None:
            yield node
            node = node.next

node1 = Node(1)
node2 = Node(2)
node3 = Node(3)

node1.next = node2
node2.next = node3
# node3.next = node1  # 别尝试

for node in node1:
    print(node.num)

print("--------------")

for node in iter(node1):
    # 对生成器对象使用 next 函数时，才开始真正运行函数本体
    # 生成器函数在被调用时并不立即执行，而是返回一个生成器对象
    print(node.num)


1
2
3
--------------
1
2
3


生成器的用法和迭代器几乎一样

但生成器本身有一个高级用法 - **send**

在生成器 yield 后，把 yield 内容变成一个值，该值可以继续赋给生成器函数里的其他变量


In [36]:
def gen(num):
    while num>0:
        temp = yield num  # yield会等send
        if temp is not None:
            num = temp
        num -= 1

g = gen(5)

first = next(g)  # next(g) 等价于 g.send(None)
print(f"first: {first}")

second = g.send(None)
print(f"second: {second}")

print(f"send:{g.send(10)}")  # 10 会赋值给 gen() 里的 temp

for i in g:
    print(i)


first: 5
second: 4
send:9
8
7
6
5
4
3
2
1
